# Experiments 63-66
**Prueba de hiperparámetros:** multi_scale / weight_decay / dropout / momentum 

Se explorarán distintos mezclas de hiperparámetros para ver si es posible mejorar los resultados obtenidos con el Mix 2 y 3.

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v4i)***
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    - **Reference:** Freezing Backbone *(10 layers)*

## Init

In [1]:
import os
import shutil
import fnmatch
import pickle

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

## Helper Functions

In [3]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [4]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [5]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [6]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [7]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [8]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Validation functions

In [9]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [10]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [11]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)

  return matrix


In [12]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)

    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [13]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f1:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

# Datasets builder

## Importing from Drive

In [16]:
!rm -rf /content/sample_data

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px		       3.5m.v4i.yolov8_blended.640px
3.5m.v3i.yolov8.640px.aug.v1	       3.5m.v4i.yolov8_blended.640px_clahe
3.5m.v3i.yolov8.640px.aug.v1.soil_aug  best_e26.pt
3.5m.v3i.yolov8.640px_clahe	       Inference
3.5m.v3i.yolov8.640px.soil_aug	       models
3.5m.v4i.yolov8.640px		       optuna_yolov8_f1_study.db
3.5m.v4i.yolov8.640px_aug5m


In [19]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 13 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8.640px_clahe',
 '3.5m.v4i.yolov8.640px',
 '3.5m.v4i.yolov8.640px_aug5m',
 '3.5m.v4i.yolov8_blended.640px',
 '3.5m.v4i.yolov8_blended.640px_clahe']

**For this experiments:** `3.5m.v4i.yolov8.640px`

In [20]:
choose_dataset = 10
index = choose_dataset - 1
model_name = os.listdir(drive_path)[index]
print("Chosen model:", model_name)

Chosen model: 3.5m.v4i.yolov8.640px


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [21]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

In [22]:
src_folder = f"/content/YOLO/{model_name}"
data = f"{src_folder}/data.yaml"
data

'/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml'

## Download model

In [14]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [15]:
# Load pretrain YOLO v8 model
model = YOLO("yolov8m.pt")

100%|██████████| 49.7M/49.7M [00:00<00:00, 122MB/s]


# Finetuning

### Optimization

In [38]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [41]:
# Garbage collection
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [ ]:
!nvidia-smi

Thu May  8 15:24:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!yolo version

8.3.129


# Experiments

## Experiment A
### *YOLOv8 Mid | Mix*

- multi_scale=True
- weight_decay=0.0015 (Explorar un - weight_decay ligeramente mayor)
- momentum=0.98
- dropout=0 (default)
**Justificación:** Mantener los parámetros exitosos de Mix 2 y ver si una regularización de weight_decay un poco más fuerte mejora aún más el rendimiento, especialmente en Precision y F½ Score.


### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015, # Superior al anterior
    #dropout=0.2,
    momentum=0.98,
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.98, mosaic=1.0, multi_scale=True, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.67G reserved, 0.33G allocated, 13.74G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.544         47.45          45.4        (1, 3, 640, 640)                    list
    25856899       158.1         2.108          38.9         72.91        (2, 3, 640, 640)                    list
    25856899       316.3         3.173         47.81         82.45        (4, 3, 640, 640)                    list
    25856899       632.5         4.958         80.17         139.8        (8, 3, 640, 640)                    list
    25856899        1265         8.563         155.2         266.4       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 14 for CUDA:0 8.69G/14.74G (59%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2086.8±669.9 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...

    200/257      12.1G      1.321     0.7195       1.08        119        544: 100%|██████████| 20/20 [00:10<00:00,  1.88it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.09it/s]

                   all        108       3472      0.567       0.48      0.473      0.155
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 100, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



200 epochs completed in 0.779 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 52.0MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:05<00:00,  1.47s/it]


                   all        108       3472      0.568      0.514        0.5      0.174
Speed: 0.3ms preprocess, 11.2ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to runs/detect/train2


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8f9beb3f10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=14,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train2',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
    

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train2


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2194.0±834.1 MB/s, size: 91.4 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.35s/it]


                   all        108       3472      0.572      0.545      0.539      0.207
Speed: 5.7ms preprocess, 23.5ms inference, 0.0ms loss, 2.3ms postprocess per image
Saving runs/detect/val2/predictions.json...
Results saved to runs/detect/val2


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val2


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val2


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4681.0
Confusion matrix:
['44.82%', '25.83%']
['29.35%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveA/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveA/


### Metrics

In [ ]:
matrix

[[2098.0, 1209.0], [1374.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4681.0

Confusion matrix:
[ 44.82% , 25.83% ]
[ 29.35% , 0.00% ]

Metrics:
- Accuracy: 0.448
- Precision: 0.634
- Recall: 0.604
- F1 Score: 0.619
- F½ Score: 0.628
- G-mean: 0.619


Comparación con Reference:

- Accuracy: Mejora (0.448 > 0.406) **+10.34%**
- Precision: Mejora (0.634 > 0.579) **+9.49%**
- Recall: Mejora (0.604 > 0.576)
- F1 Score: Mejora (0.619 > 0.577)
- F½ Score: Mejora (0.628 > 0.578) **+8.65%**
- G-mean: Mejora (0.619 > 0.577)

Hubo mejoras significativas en todas las métricas clave en comparación con el modelo de referencia. Exp A mostró un rendimiento consistentemente superior.


Compararción con Mix 1 (59):

- Accuracy: Mejora (0.448 > 0.433)
- Precision: Mejora (0.634 > 0.611)
- Recall: Constante (0.604 ~ 0.597)
- F1 Score: Mejora (0.619 > 0.604)
- F½ Score: Mejora (0.628 > 0.608)
- G-mean: Mejora (0.619 > 0.604)

Conclusión: Exp A obtuvo mejores resultados que Mix 1 (59) en todas las métricas.

-----
## Experiment B
### *YOLOv8 Mid | Mix*
- multi_scale=True
- weight_decay=0.001
- momentum=0.99 (Explorar un momentum aún más cercano a 1)
- dropout=0 (default)
**Justificación:** Mantener el weight_decay de Mix 2 y probar si un momentum extremadamente alto puede ofrecer beneficios adicionales para la convergencia y el rendimiento final en las métricas objetivo.

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.001,
    #dropout=0.2,
    momentum=0.99, # Superior al anterior
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 0.53G reserved, 0.33G allocated, 13.87G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.715         55.11          51.7        (1, 3, 640, 640)                    list
    25856899       158.1         2.187         35.53         69.37        (2, 3, 640, 640)                    list
    25856899       316.3         3.253         59.53         109.5        (4, 3, 640, 640)                    list
    25856899       632.5         5.018         83.48         141.6        (8, 3, 640, 640)                    list
    25856899        1265         8.401         159.1         271.8       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 14 for CUDA:0 8.45G/14.74G (57%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1971.6±829.6 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...

    155/258      11.9G      1.488     0.8288       1.11        126        512: 100%|██████████| 20/20 [00:10<00:00,  1.89it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:01<00:00,  2.12it/s]

                   all        108       3472       0.55       0.47      0.455      0.149
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 55, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



155 epochs completed in 0.602 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 52.0MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:07<00:00,  1.76s/it]


                   all        108       3472      0.541      0.484      0.494      0.176
Speed: 0.2ms preprocess, 11.4ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to runs/detect/train3


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8f869c6950>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=14,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train3',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
    

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train3


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1932.5±1034.7 MB/s, size: 96.7 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.35s/it]


                   all        108       3472      0.572      0.499      0.529      0.208
Speed: 5.8ms preprocess, 23.4ms inference, 0.0ms loss, 1.8ms postprocess per image
Saving runs/detect/val3/predictions.json...
Results saved to runs/detect/val3


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val3


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val3


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4585.0
Confusion matrix:
['41.83%', '24.27%']
['33.89%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveB/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveB/


### Metrics

In [ ]:
matrix

[[1918.0, 1113.0], [1554.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4585.0

Confusion matrix:
[ 41.83% , 24.27% ]
[ 33.89% , 0.00% ]

Metrics:
- Accuracy: 0.418
- Precision: 0.633
- Recall: 0.552
- F1 Score: 0.590
- F½ Score: 0.615
- G-mean: 0.591


Comparación con Reference:

- Accuracy: Mejora (0.418 > 0.406)
- Precision: Mejora (0.633 > 0.579) **+9.33%**
- Recall: Empeoramiento (0.552 < 0.576) **-4.17%**
- F1 Score: Constante (0.590 ~ 0.577)
- F½ Score: Mejora (0.615 > 0.578) **+6.40%**
- G-mean: Constante (0.591 ~ 0.577)

Compararción con Mix 1 (59):

- Accuracy: Empeora (0.418 < 0.433)
- Precision: Mejora (0.633 > 0.611)
- Recall: Empeora (0.552 < 0.597) **-7.54%**
- F1 Score: Empeora (0.590 < 0.604)
- F½ Score: Mejora (0.615 > 0.608)
- G-mean: Mejora (0.604 > 0.591)

-----
## Experiment C
### *YOLOv8 Mid | Mix*

Best mix + dropout

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015, # Superior al anterior
    dropout=0.1,  # Inferior al anterior
    momentum=0.99, # Superior al anterior
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 8.88G reserved, 0.85G allocated, 5.02G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.550         34.14         30.49        (1, 3, 640, 640)                    list
    25856899       158.1         2.143         32.15         42.25        (2, 3, 640, 640)                    list
    25856899       316.3         3.169         40.97         76.57        (4, 3, 640, 640)                    list
    25856899       632.5         4.991         85.74         142.3        (8, 3, 640, 640)                    list
    25856899        1265         8.475         162.7         279.8       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 3 for CUDA:0 12.33G/14.74G (84%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1887.3±290.0 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


    190/190      4.76G      1.445     0.8133      1.151         45        384:  61%|██████    | 55/90 [00:09<00:05,  5.84it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 18/18 [00:02<00:00,  8.84it/s]


                   all        108       3472      0.571      0.487      0.481      0.164

190 epochs completed in 1.001 hours.
Optimizer stripped from runs/detect/train4/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train4/weights/best.pt, 52.0MB

Validating runs/detect/train4/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 18/18 [00:03<00:00,  4.57it/s]


                   all        108       3472      0.557      0.495      0.503      0.179
Speed: 0.6ms preprocess, 11.2ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/train4


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8fabc2bc10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=3,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train4',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.1,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
     

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train4


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1815.5±319.5 MB/s, size: 99.7 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.55s/it]


                   all        108       3472      0.596      0.497      0.541      0.215
Speed: 5.1ms preprocess, 23.7ms inference, 0.0ms loss, 2.9ms postprocess per image
Saving runs/detect/val4/predictions.json...
Results saved to runs/detect/val4


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val4


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val4


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4458.0
Confusion matrix:
['42.87%', '22.12%']
['35.02%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveC/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveC/


### Metrics

In [ ]:
matrix

[[1911.0, 986.0], [1561.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4458.0

Confusion matrix:
[ 42.87% , 22.12% ]
[ 35.02% , 0.00% ]

Metrics:
- Accuracy: 0.429
- Precision: 0.660
- Recall: 0.550
- F1 Score: 0.600
- F½ Score: 0.634
- G-mean: 0.603


Comparación de EXP C con Reference (Base = Reference)

Accuracy: ((0.429 - 0.406) / 0.406) * 100 ≈ +5.66% (Mejora)
Precision: ((0.660 - 0.579) / 0.579) * 100 ≈ +13.99% (Mejora)
Recall: ((0.550 - 0.576) / 0.576) * 100 ≈ -4.51% (Empeoramiento)
F1 Score: ((0.600 - 0.577) / 0.577) * 100 ≈ +3.99% (Mejora)
F½ Score: ((0.634 - 0.578) / 0.578) * 100 ≈ +9.69% (Mejora)
G-mean: ((0.603 - 0.577) / 0.577) * 100 ≈ +4.51% (Mejora)
Conclusión vs Reference: EXP C mostró mejoras significativas en Accuracy, Precision, F1 Score, F½ Score y G-mean en comparación con la Referencia, pero tuvo un empeoramiento en Recall.

Comparación de EXP C con Mix 1 (59) (Base = Mix 1)

Accuracy: ((0.429 - 0.433) / 0.433) * 100 ≈ -0.92% (Constante, variación < 2%)
Precision: ((0.660 - 0.611) / 0.611) * 100 ≈ +7.97% (Mejora)
Recall: ((0.550 - 0.597) / 0.597) * 100 ≈ -7.87% (Empeoramiento)
F1 Score: ((0.600 - 0.604) / 0.604) * 100 ≈ -0.66% (Constante, variación < 2%)
F½ Score: ((0.634 - 0.608) / 0.608) * 100 ≈ +4.28% (Mejora)
G-mean: ((0.603 - 0.604) / 0.604) * 100 ≈ -0.17% (Constante, variación < 2%)


-----
## Experiment D
### *YOLOv8 Mid | Mix*

Best mix w/o dropout

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 1 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
model = YOLO("yolov8m.pt")
history = model.train(
    data=data,
    epochs=500,
    val=True,
    imgsz=640,
    batch=-1,
    patience=100,
    time = time,
    multi_scale=True,
    weight_decay=0.0015, # Superior al anterior
    #dropout=0.1,
    momentum=0.99, # Superior al anterior
)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.99, mosaic=1.0, multi_scale=True, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (Tesla T4) 14.74G total, 7.04G reserved, 0.74G allocated, 6.96G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output


    25856899       79.07         1.556         38.24         30.69        (1, 3, 640, 640)                    list
    25856899       158.1         2.095         31.83         42.92        (2, 3, 640, 640)                    list
    25856899       316.3         3.196         40.43          77.2        (4, 3, 640, 640)                    list
    25856899       632.5         4.983         83.93         140.4        (8, 3, 640, 640)                    list
    25856899        1265         8.623         159.3         274.2       (16, 3, 640, 640)                    list
AutoBatch: Using batch-size 6 for CUDA:0 11.78G/14.74G (80%) ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1528.0±392.4 MB/s, size: 89.0 KB)


train: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

...

    155/231      6.73G      1.616        0.9      1.186        135        544: 100%|██████████| 45/45 [00:11<00:00,  3.79it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:01<00:00,  4.53it/s]

                   all        108       3472      0.577      0.507      0.497      0.163
EarlyStopping: Training stopped early as no improvement observed in last 100 epochs. Best results observed at epoch 55, best model saved as best.pt.
To update EarlyStopping(patience=100) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



155 epochs completed in 0.672 hours.
Optimizer stripped from runs/detect/train5/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train5/weights/best.pt, 52.0MB

Validating runs/detect/train5/weights/best.pt...
Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 9/9 [00:06<00:00,  1.49it/s]


                   all        108       3472      0.554      0.504      0.502       0.18
Speed: 0.4ms preprocess, 12.5ms inference, 0.0ms loss, 5.0ms postprocess per image
Results saved to runs/detect/train5


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8f864dd290>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8.640px/data.yaml',
          epochs=500,
          time=1,
          patience=100,
          batch=6,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train5',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=True,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=0.7,
     

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train5


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.129 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1882.9±556.3 MB/s, size: 82.7 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:06<00:00,  3.41s/it]


                   all        108       3472      0.583      0.518      0.539      0.214
Speed: 5.5ms preprocess, 23.6ms inference, 0.0ms loss, 2.5ms postprocess per image
Saving runs/detect/val5/predictions.json...
Results saved to runs/detect/val5


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val5


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val5


In [ ]:
matrix = gimme_metrics(results)

Total objects detected: 4604.0
Confusion matrix:
['42.40%', '24.59%']
['33.01%', '0.00%']


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/saveD/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/saveD/


### Metrics

In [ ]:
matrix

[[1952.0, 1132.0], [1520.0, 0.0]]

In [ ]:
# Setting values from CM graph
TP = matrix[0][0]
FP = matrix[0][1]
FN = matrix[1][0]
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4604.0

Confusion matrix:
[ 42.40% , 24.59% ]
[ 33.01% , 0.00% ]

Metrics:
- Accuracy: 0.424
- Precision: 0.633
- Recall: 0.562
- F1 Score: 0.595
- F½ Score: 0.617
- G-mean: 0.597


Comparación con EXP C:
- Accuracy: Constante (0.424 ~ 0.429)
- Precision: Empeora (0.633 < 0.660) **-4.09%**
- Recall: Constante (0.562 ~ 0.550)
- F1 Score: Constante (0.595 ~ 0.600)
- F½ Score: Empeora (0.617 < 0.634)
- G-mean: Constante (0.597 ~ 0.603)

Podemos apreciar el impacto negativo de haber removido el dropout, observamos que EXP D tuvo un empeoramiento notable en Precision y F½ Score. Sin embargo, mostró una ligera mejora en Recall (+2.18%).

Comparación con Mix 3 (61):
- Accuracy: Constante (0.424 vs 0.433)
- Precision: Constante (0.633 vs 0.636
- Recall: Empeora (0.562 vs 0.576) -2.43%
- F1 Score: Constante (0.595 vs 0.605)
- F½ Score: Constante (0.617 vs 0.623)
- G-mean: Constante (0.597 vs 0.605)

**Conclusión:**  Podemos concluir que agregar un dropout pequeño mejora el entrenamiento.

# Comparación final

| Notebook        | **Accuracy** | **Precision** | **Recall** | **F1 Score** | **F½ Score** | **G-mean** | referencia |
|-----------------|--------------|---------------|------------|--------------|--------------|------------|------------|
| multi-scale (55)| 0.406        | 0.579         | 0.576      | 0.577        | 0.578        | 0.577      |            |
| Wight decay (56)| 0.419        | 0.648         | 0.542      | 0.590        | 0.623        | 0.592      |            |
| Dropout (57)| 0.417        | 0.638         | 0.547      | 0.589        | 0.618        | 0.591      |            |
| Momentum (58)| 0.416        | 0.644         | 0.540      | 0.587        | 0.620        | 0.589      |            |
| |
| Mix 1 (59)| 0.426        | 0.630         | 0.569      | 0.598        | 0.617        | 0.599      |            |
| Mix 2 (60) | 0.433        | 0.611         | 0.597 | 0.604        | 0.608        | 0.604      |            |
| Mix 3 (61) | 0.426        | 0.659 | 0.546      | 0.597        | 0.633        | 0.600      |            |
| Mix 4 (62)      | 0.433        | 0.636         | 0.576      | 0.605        | 0.623        | 0.605      |            |
| |
| Exp A    | 0.438        | 0.635         | 0.584      | 0.609        | 0.625        | 0.609      |            |
| Exp B    | **0.448** | 0.634         | **0.604**      | **0.619** | 0.628 | **0.619** |            |
| Exp C    | 0.429        | **0.660**         | 0.550      | 0.600        | **0.634** | 0.603      |            |
| Exp D    | 0.424        | 0.633         | 0.562      | 0.595        | 0.617        | 0.597      |            |

| Notebook        | Accuracy | Precision | Recall | F1 Score | F½ Score | G-mean | multi_scale | weight_decay | dropout | momentum |
|-----------------|----------|-----------|--------|----------|----------|--------|-------------|--------------|---------|----------|
| multi-scale (55)| 0.406    | 0.579     | 0.576  | 0.577    | 0.578    | 0.577  | (default)   | (default)    | (default)| (default)|
| Wight decay (56)| 0.419    | 0.648     | 0.542  | 0.590    | 0.623    | 0.592  | (default)   | aplicado     | (default)| (default)|
| Dropout (57)    | 0.417    | 0.638     | 0.547  | 0.589    | 0.618    | 0.591  | (default)   | (default)    | aplicado| (default)|
| Momentum (58)   | 0.416    | 0.644     | 0.540  | 0.587    | 0.620    | 0.589  | (default)   | (default)    | (default)| aplicado |
| Mix 1 (59)      | 0.426    | 0.630     | 0.569  | 0.598    | 0.617    | 0.599  | aplicado    | aplicado     | aplicado| aplicado |
| Mix 2 (60)      | 0.433    | 0.611     | 0.597  | 0.604    | 0.608    | 0.604  | (default)   | (default)    | (default)| (default)|
| Mix 3 (61)      | 0.426    | 0.659     | 0.546  | 0.597    | 0.633    | 0.600  | (default)   | (default)    | (default)| (default)|
| Mix 4 (62)      | 0.433    | 0.636     | 0.576  | 0.605    | 0.623    | 0.605  | aplicado    | (default)    | aplicado| (default)|
| Exp A           | 0.438    | 0.635     | 0.584  | 0.609    | 0.625    | 0.609  | (default)   | (default)    | (default)| (default)|
| Exp B           | 0.448    | 0.634     | 0.604  | 0.619    | 0.628    | 0.619  | (default)   | (default)    | (default)| (default)|
| Exp C           | 0.429    | 0.660     | 0.550  | 0.600    | 0.634    | 0.603  | (default)   | (default)    | (default)| (default)|
| Exp D           | 0.424    | 0.633     | 0.562  | 0.595    | 0.617    | 0.597  | (default)   | (default)    | (default)| (default)|